# Fine-tune KumoTabular Classification

Fine-tunes a pretrained KumoTabular model on the BNP Paribas Cardif Claims training data. Saves the best checkpoint to a UC Volume and logs metrics to MLflow.

**Compute:** GPU (A10)

In [0]:
%pip install git+https://github.com/NVIDIA/structured-data-models.git --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

SCHEMA = "default"
VOLUME_NAME = "finetuned_models"
TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.bnp_paribas_cardif_train"
TARGET_COL = "target"
ID_COL = "ID"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks-finetune"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"Volume: {volume_path}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

In [0]:
import gc, time

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

import mlflow
import torch
import torch.nn.functional as F
import sdm
import sdm.processing as sp

In [0]:
# Load training data from UC
train_pdf = spark.table(TRAIN_TABLE).toPandas()
train_fit_pdf = train_pdf.dropna(subset=[TARGET_COL])

# Feature columns (exclude ID and target)
all_feature_cols = [c for c in train_fit_pdf.columns if c not in (ID_COL, TARGET_COL)]
feature_cols = all_feature_cols

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Train: {len(train_fit_pdf):,} rows")
print(f"Features: {len(feature_cols)} of {len(all_feature_cols)} total")
print(f"Device: {device}")

In [0]:
# ── Fine-tuning configuration ───────────────────────────────────────
FT_MODEL_SIZE = "small"
FT_LR = 1e-5
FT_MAX_EPOCHS = 50
FT_PATIENCE = 10
FT_CONTEXT_SIZE = 3000 #2048
FT_QUERY_SIZE = 512
FT_FEATURE_PERCENTAGE_TRAIN = 0.5
FT_NUM_ESTIMATORS_TRAIN = 1
FT_NUM_ESTIMATORS_VAL = 1
FT_EVAL_CONTEXT_SIZE = 16384
FT_MIN_DELTA = 1e-4
FT_WEIGHT_DECAY = 0.01
FT_WARMUP_PROPORTION = 0.1          # fraction of total steps for linear warmup
FT_SEED = 0

torch.manual_seed(FT_SEED)

# ── Prepare data ──────────────────────────────────────────────────
ft_cols = feature_cols + [TARGET_COL]
ft_df = train_fit_pdf[ft_cols].reset_index(drop=True)

stypes = sdm.infer_stypes(ft_df, overrides={TARGET_COL: "categorical"})
table = sdm.TableTensor.from_pandas(ft_df, stypes=stypes, device=device)

train_idx, val_idx = train_test_split(
    torch.arange(len(ft_df)).numpy(),
    test_size=0.2,
    random_state=FT_SEED,
    stratify=ft_df[TARGET_COL],
)
train_table = table[torch.from_numpy(train_idx).to(device)]
val_table = table[torch.from_numpy(val_idx).to(device)]
del table
gc.collect()

# Full-coverage chunking: compute steps from data
chunk_size = FT_CONTEXT_SIZE + FT_QUERY_SIZE
steps_per_epoch = max(1, len(train_idx) // chunk_size)

print(f"Train: {len(train_idx):,} rows | Val: {len(val_idx):,} rows")
print(f"Features: {len(feature_cols)} | Device: {device}")
print(f"Chunking: {len(train_idx):,} rows / {chunk_size} = {steps_per_epoch} steps/epoch (full coverage)")

# ── Model & recipe ────────────────────────────────────────────────
model = sdm.models.KumoTabular(
    task="classification", size=FT_MODEL_SIZE, device=device, pretrained=True,
)

train_recipe = model.default_recipe()
train_recipe.target = sp.StypeDispatch(
    categorical=[
        sp.AlignCategories(),
        sp.ShuffleCategories(method="shift"),
    ],
    numerical=sp.Standardize(),
)
train_recipe.output = sp.Identity()


def evaluate():
    """Evaluate full val set in one pass, random context each call."""
    model.eval()
    ctx_idx = torch.randperm(train_table.size(0), device=device)[:FT_EVAL_CONTEXT_SIZE]
    ctx = train_table[ctx_idx]
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.float16):
            out = model(
                x_context=ctx.drop_columns(TARGET_COL),
                y_context=ctx[TARGET_COL],
                x_query=val_table.drop_columns(TARGET_COL),
                num_estimators=FT_NUM_ESTIMATORS_VAL,
            )
    prob, y = sdm.evaluation.to_class_indices(
        out, val_table[TARGET_COL], missing_score=0.0,
    )
    prob = torch.softmax(prob, dim=-1)
    y_pred = prob.argmax(dim=-1).cpu().numpy()
    y_true = y.cpu().numpy()
    y_prob = prob[:, -1].cpu().numpy()
    del out, prob, ctx
    torch.cuda.empty_cache()
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, average="binary")),
        "log_loss": float(log_loss(y_true, y_prob, labels=[0, 1])),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
    }


# ── Training loop with MLflow ─────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda')

# Cosine-with-warmup LR scheduler (built-in PyTorch)
total_steps = FT_MAX_EPOCHS * steps_per_epoch
warmup_steps = int(total_steps * FT_WARMUP_PROPORTION)

scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[
    torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-8, end_factor=1.0, total_iters=warmup_steps),
    torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps),
], milestones=[warmup_steps])
print(f"LR schedule: warmup {warmup_steps} steps, cosine decay over {total_steps} total steps")

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="kumo_classification_finetuning_process"):
    run_id = mlflow.active_run().info.run_id
    ckpt_path = f"{volume_path}/kumo_tabular_{FT_MODEL_SIZE}_classification_finetuned_{run_id}.pt"

    mlflow.log_params({
        "vendor": "kumo",
        "model_type": "KumoTabular",
        "model_size": FT_MODEL_SIZE,
        "task": "classification",
        "dataset": "bnp_paribas_cardif",
        "lr": FT_LR,
        "max_epochs": FT_MAX_EPOCHS,
        "patience": FT_PATIENCE,
        "steps_per_epoch": steps_per_epoch,
        "chunk_size": chunk_size,
        "context_size": FT_CONTEXT_SIZE,
        "query_size": FT_QUERY_SIZE,
        "feature_percentage_train": FT_FEATURE_PERCENTAGE_TRAIN,
        "num_estimators_train": FT_NUM_ESTIMATORS_TRAIN,
        "num_estimators_val": FT_NUM_ESTIMATORS_VAL,
        "eval_context_size": FT_EVAL_CONTEXT_SIZE,
        "early_stop_metric": "roc_auc",
        "min_delta": FT_MIN_DELTA,
        "weight_decay": FT_WEIGHT_DECAY,
        "warmup_proportion": FT_WARMUP_PROPORTION,
        "n_features": len(feature_cols),
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "seed": FT_SEED,
        "device": device,
    })

    # Baseline before any training
    val_metrics = evaluate()
    print(f"epoch=0/{FT_MAX_EPOCHS} val_acc={val_metrics['accuracy']:.4f} val_f1={val_metrics['f1']:.4f} val_logloss={val_metrics['log_loss']:.4f} val_auc={val_metrics['roc_auc']:.4f}")
    mlflow.log_metrics({f"ft_per_epoch_val_{k}": v for k, v in val_metrics.items()}, step=0)

    best_val_metrics = val_metrics.copy()
    best_val_roc_auc = val_metrics["roc_auc"]
    epochs_without_improvement = 0
    torch.save(model.state_dict(), ckpt_path)

    gc.collect()
    torch.cuda.empty_cache()

    t_start = time.time()
    for epoch in range(1, FT_MAX_EPOCHS + 1):
        t_epoch = time.time()
        model.train()
        total_loss = 0.0

        # Shuffle all training indices and chunk for full coverage
        perm = torch.randperm(train_table.size(0), device=device)
        for step in range(steps_per_epoch):
            start = step * chunk_size
            chunk_idx = perm[start : start + chunk_size]
            context = train_table[chunk_idx[:FT_CONTEXT_SIZE]]
            query = train_table[chunk_idx[FT_CONTEXT_SIZE:]]

            # Random feature subsetting each step
            n_feat = len(feature_cols)
            n_keep = max(1, int(n_feat * FT_FEATURE_PERCENTAGE_TRAIN))
            feat_idx = torch.randperm(n_feat)[:n_keep].sort().values.tolist()
            drop_cols = [feature_cols[i] for i in range(n_feat) if i not in feat_idx] + [TARGET_COL]

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = model(
                    x_context=context.drop_columns(drop_cols),
                    y_context=context[TARGET_COL],
                    x_query=query.drop_columns(drop_cols),
                    recipe=train_recipe,
                    num_estimators=FT_NUM_ESTIMATORS_TRAIN,
                )[0]
                logits, y = sdm.evaluation.to_class_indices(
                    out, query[TARGET_COL], missing_score=-torch.inf,
                )
                loss = F.cross_entropy(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += float(loss.detach())
            del context, query, out, logits, y, loss
            torch.cuda.empty_cache()

        del perm
        avg_loss = total_loss / steps_per_epoch
        gc.collect()
        torch.cuda.empty_cache()

        val_metrics = evaluate()

        gc.collect()
        torch.cuda.empty_cache()

        epoch_time = time.time() - t_epoch
        print(
            f"epoch={epoch}/{FT_MAX_EPOCHS} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_f1={val_metrics['f1']:.4f} "
            f"val_logloss={val_metrics['log_loss']:.4f} "
            f"val_auc={val_metrics['roc_auc']:.4f} "
            f"train_loss={avg_loss:.4f} "
            f"time={epoch_time:.1f}s"
        )
        mlflow.log_metrics({f"ft_per_epoch_val_{k}": v for k, v in val_metrics.items()}, step=epoch)
        mlflow.log_metric("ft_per_epoch_train_loss", avg_loss, step=epoch)

        if val_metrics["roc_auc"] > best_val_roc_auc + FT_MIN_DELTA:
            best_val_roc_auc = val_metrics["roc_auc"]
            best_val_metrics = val_metrics.copy()
            epochs_without_improvement = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  → New best (roc_auc={best_val_roc_auc:.4f})! Saved checkpoint.")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= FT_PATIENCE:
                print(f"  → Early stopping after {FT_PATIENCE} epochs without improvement.")
                break

    ft_time = time.time() - t_start

    best_state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(best_state)

    mlflow.log_metrics({
        "ft_best_val_acc": best_val_metrics["accuracy"],
        "ft_best_val_f1": best_val_metrics["f1"],
        "ft_best_val_log_loss": best_val_metrics["log_loss"],
        "ft_best_val_roc_auc": best_val_metrics["roc_auc"],
        "ft_finetune_time_s": ft_time,
        "ft_epochs_completed": min(epoch, FT_MAX_EPOCHS),
    })
    mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")

print(f"\n✓ Fine-tuning complete (run {run_id})")
print(f"  Best val accuracy: {best_val_metrics['accuracy']:.4f}")
print(f"  Best val f1:       {best_val_metrics['f1']:.4f}")
print(f"  Best val log_loss: {best_val_metrics['log_loss']:.4f}")
print(f"  Time: {ft_time:.1f}s")
print(f"  Checkpoint: {ckpt_path}")